## Workflow for Neosurf-on-Neosurf MaSIF search

In [10]:
import os
import pandas as pd
import sys
import numpy as np
import subprocess

repo_root = !git rev-parse --show-toplevel
repo_root = repo_root[0]
os.chdir(repo_root)


# ----- Source code and scripts ------
# Add source_dir to python PATH
source_dir = os.path.join(repo_root, 'masif_seed_search/source')
sys.path.insert(0, source_dir)

prepare_input_py = os.path.join(repo_root, 'scripts/python/prepare_input.py')

# Settings:
N_ARRAY_JOBS = 100
EVOEF2_BIN = os.path.join(repo_root, "EvoEF2/EvoEF2")


# ----- Directories ------
data_dir = os.path.join(repo_root, 'data')

# ----- Input ------
# .csv about the seed complexes - use all human reference proteome liganded pdbs
seed_list_csv = os.path.join(data_dir, "human_reference_proteome_liganded_pdbs/human_reference_proteome_pdb_ligands_split.csv")

# Directory to store input .pdb and .sdf
input_dir = os.path.join(data_dir, 'input')
os.makedirs(input_dir, exist_ok=True)

# ----- Processing directory ------
processing_dir = os.path.join(data_dir, 'processing')
os.makedirs(processing_dir, exist_ok=True)

# 1. Prepare input files
prep_input_proc_dir = os.path.join(processing_dir, '1_prep_input')
os.makedirs(prep_input_proc_dir, exist_ok=True)

# 2. Run MaSIF preprocessing
masif_preprocess_proc_dir = os.path.join(processing_dir, '2_masif_preprocess')
os.makedirs(masif_preprocess_proc_dir, exist_ok=True)

# 3. Run MaSIF search
masif_search_proc_dir = os.path.join(processing_dir, '3_masif_search')
os.makedirs(masif_search_proc_dir, exist_ok=True)

# ----- Output ------
# Directory to write preprocessing files
preprocess_dir = os.path.join(data_dir, 'preprocess')

# Directory to write masif-search output
masif_search_out_dir = os.path.join(data_dir, 'masif_search')
os.makedirs(masif_search_out_dir, exist_ok=True)


___
### Step 1 - preprocess all targets
1. Preprocess targets with ligands in nico_targets.csv
2. Preprocess VHL and CRBN with ligands

In [11]:
df_seed = pd.read_csv(seed_list_csv)

df_seed.head()

,uniprot_id,gene_name,recommendedName,pdb_id,protein_chain,ligand_chain,ligand_code,ligand_name,smiles,percent_intracellular,formula,mw,qed,num_carbon,num_N_O,uniprot_id_count,split
0,A1L3X0,ELOVL7,Very long chain fatty acid elongase 7,6Y7F,A,A,OFN,"~{S}-[2-[3-[[(2~{R})-4-[[[(2~{R},3~{S},4~{R},5...",CCCCCCCCCCCCCCCCCC(=O)CC(=O)SCCNC(=O)CCNC(=O)[...,0.213523,C41H72N7O18P3S,1075.386739,0.024703,41.0,25.0,1.0,seed
1,A1L3X0,ELOVL7,Very long chain fatty acid elongase 7,6Y7F,A,A,37X,Octyl Glucose Neopentyl Glycol,CCCCCCC(CCCCCC)(CO[C@@H]1[C@H]([C@@H]([C@H]([C...,0.213523,C27H52O12,568.345877,0.099222,27.0,12.0,3.0,seed
2,A0FGR8,ESYT2,Extended synaptotagmin-2,4P42,A,A,EGC,"2-(2-{2-[2-(2-{2-[2-(2-{2-[4-(1,1,3,3-TETRAMET...",CC(C)(C)CC(C)(C)c1ccc(cc1)OCCOCCOCCOCCOCCOCCOC...,1.000000,C32H58O10,602.402998,0.135862,32.0,10.0,1.0,seed
3,A4D1P6,WDR91,WD repeat-containing protein 91,8SHJ,A,A,ZI8,N-[3-(4-chlorophenyl)oxetan-3-yl]-4-[(3S)-3-hy...,c1cc(ccc1C(=O)NC2(COC2)c3ccc(cc3)Cl)N4CC[C@@H]...,1.000000,C20H21ClN2O3,372.124070,0.865528,20.0,5.0,1.0,seed
4,A4D1P6,WDR91,WD repeat-containing protein 91,8T55,C,C,ZI3,N-[3-(4-chlorophenyl)oxetan-3-yl]-1-propanoyl-...,CCC(=O)N1CCCc2c1cccc2C(=O)NC3(COC3)c4ccc(cc4)Cl,1.000000,C22H23ClN2O3,398.139720,0.854092,22.0,5.0,1.0,seed


In [13]:
# Prepare input files for a single complex
df_input_subset = pd.DataFrame(
    [
        {
            "pdb_id": "8VLB",
            "protein_chain": "A",
            "ligand_chain": "A",
            "ligand_code": "3JF"
        }
    ]
)

df_input_subset.to_csv(os.path.join(data_dir, f"prepare_input.csv"), index=False)

cmd = [
    "python",
    prepare_input_py,
    "--input_csv", os.path.join(data_dir, f"prepare_input.csv"),
    "--outdir", input_dir,
    "--out_csv", os.path.join(data_dir, f"prepare_input_out.csv"),
    "--evoef2_bin", EVOEF2_BIN
]
print(cmd)
# subprocess.run(cmd)


['python', '/scratch/ymeng/Neosurf_Neosurf/scripts/python/prepare_input.py', '--input_csv', '/scratch/ymeng/Neosurf_Neosurf/data/prepare_input.csv', '--outdir', '/scratch/ymeng/Neosurf_Neosurf/data/input', '--out_csv', '/scratch/ymeng/Neosurf_Neosurf/data/prepare_input_out.csv', '--evoef2_bin', '/scratch/ymeng/Neosurf_Neosurf/EvoEF2/EvoEF2']


Preparing structures:   0%|          | 0/1 [00:00<?, ?it/s][13:33:58] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.


Wrote /scratch/ymeng/Neosurf_Neosurf/data/prepare_input_out.csv  (1/1 rows succeeded)


[13:33:58] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
Preparing structures: 100%|██████████| 1/1 [00:02<00:00,  2.73s/it]


CompletedProcess(args=['python', '/scratch/ymeng/Neosurf_Neosurf/scripts/python/prepare_input.py', '--input_csv', '/scratch/ymeng/Neosurf_Neosurf/data/prepare_input.csv', '--outdir', '/scratch/ymeng/Neosurf_Neosurf/data/input', '--out_csv', '/scratch/ymeng/Neosurf_Neosurf/data/prepare_input_out.csv', '--evoef2_bin', '/scratch/ymeng/Neosurf_Neosurf/EvoEF2/EvoEF2'], returncode=0)

In [14]:
# Prepare input files
input_subset_dir = os.path.join(prep_input_proc_dir, "input_subsets")
os.makedirs(input_subset_dir, exist_ok=True)
output_subset_dir = os.path.join(prep_input_proc_dir, "output_subsets")
os.makedirs(output_subset_dir, exist_ok=True)

# Split df_seed into N_ARRAY_JOBS chunks
df_seed_subsets = np.array_split(df_seed, N_ARRAY_JOBS)

# Write each subset to a separate file
for i, df_seed_subset in enumerate(df_seed_subsets):
    df_seed_subset.to_csv(os.path.join(input_subset_dir, f"input_{i+1}.csv"), index=False)

# Submit slurm array job: task k reads input_k.csv, writes output_k.csv
cmd = [
    "sbatch",
    f"--array=1-{N_ARRAY_JOBS}",
    "scripts/slurm/prepare_input_array.sh",
    input_subset_dir,
    input_dir,
    output_subset_dir,
    EVOEF2_BIN,
]
subprocess.run(cmd, check=True)

/home/ymeng/miniconda3/envs/MaSIF/lib/python3.11/site-packages/numpy/_core/fromnumeric.py:54: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


Submitted batch job 57176911


sbatch: [ESTIMATION] The estimated cost of this job is CHF 2.20
sbatch: ╭──────────────────────────────┬─────────────┬─────────────┬─────────────╮
sbatch: │ [in CHF]                     │ Capping     │ Consumed    │ Queued ¹⁾ ²⁾│
sbatch: ├──────────────────────────────┼─────────────┼─────────────┼─────────────┤
sbatch: │ username : ymeng             │ 0           │ 72.4        │ 2.2         │
sbatch: ├──────────────────────────────┼─────────────┼─────────────┼─────────────┤
sbatch: │ account : upthomae           │ 10,000      │ 87.15       │ 2.2         │
sbatch: ╰──────────────────────────────┴─────────────┴─────────────┴─────────────╯
sbatch: ¹⁾ Estimated cost of the queued jobs and this job
sbatch: ²⁾ Queued jobs costs are based on its walltime (option --time)


CompletedProcess(args=['sbatch', '--array=1-100', 'scripts/slurm/prepare_input_array.sh', '/scratch/ymeng/Neosurf_Neosurf/data/processing/1_prep_input/input_subsets', '/scratch/ymeng/Neosurf_Neosurf/data/input', '/scratch/ymeng/Neosurf_Neosurf/data/processing/1_prep_input/output_subsets', '/scratch/ymeng/Neosurf_Neosurf/EvoEF2/EvoEF2'], returncode=0)

In [15]:
# Gather all output .csv files into a single df_input_prepared
df_input_prepared = pd.DataFrame()
for i in range(N_ARRAY_JOBS):
    csv_path = os.path.join(output_subset_dir, f"output_{i+1}.csv")
    if os.path.exists(csv_path):
        df_input_prepared = pd.concat([df_input_prepared, pd.read_csv(csv_path)])
    else:
        print(f"Warning: {csv_path} does not exist")

print(f"df_input_prepared.shape: {df_input_prepared.shape}")

# Split into success and failed rows
df_preprocess_manifest = df_input_prepared[
    df_input_prepared['pdb_path'].notna() & df_input_prepared['ligand_path'].notna()
]
df_input_failed = df_input_prepared[
    df_input_prepared['pdb_path'].isna() | df_input_prepared['ligand_path'].isna()
]
print(f"Successfully prepared {df_preprocess_manifest.shape[0]} complexes.")
print(f"Failed to prepare input files for {df_input_failed.shape[0]} complexes.")
print(f"Failed entries:")
df_input_failed.head()


df_input_prepared.shape: (28153, 21)
Successfully prepared 27934 complexes.
Failed to prepare input files for 219 complexes.
Failed entries:


,uniprot_id,gene_name,recommendedName,pdb_id,protein_chain,ligand_chain,ligand_code,ligand_name,smiles,percent_intracellular,...,mw,qed,num_carbon,num_N_O,uniprot_id_count,split,pdb_path,target,ligand,ligand_path
21,O00206,TLR4,Toll-like receptor 4,3ULA,A,A,E55,3-O-DECYL-2-DEOXY-6-O-{2-DEOXY-3-O-[(3R)-3-MET...,CCCCCCCCCCCC(=O)CC(=O)N[C@@H]1[C@H]([C@@H]([C@...,0.0,...,1312.843003,0.012884,66.0,21.0,2.0,seed,NaN,NaN,NaN,NaN
184,O00506,STK25,Serine/threonine-protein kinase 25,4NZW,B,B,J60,"5-[(E)-(5-CHLORO-2-OXO-1,2-DIHYDRO-3H-INDOL-3-...",CCN(CC)CCNC(=O)c1c(c([nH]c1C)\C=C/2\c3cc(ccc3N...,1.0,...,414.182254,0.602034,22.0,6.0,2.0,seed,NaN,NaN,NaN,NaN
209,O00482,NR5A2,Nuclear receptor subfamily 5 group A member 2,9SMQ,A,A,A1JOT,3-[5-phenyl-1-[3-(trifluoromethyl)phenyl]pyraz...,c1ccc(cc1)c2cc(nn2c3cccc(c3)C(F)(F)F)CCC(=O)O,1.0,...,360.108562,0.724829,19.0,4.0,1.0,seed,NaN,NaN,NaN,NaN
257,O14965,AURKA,Aurora kinase A,3UO4,A,A,0C0,4-{[4-(biphenyl-2-ylamino)pyrimidin-2-yl]amino...,c1ccc(cc1)c2ccccc2Nc3ccnc(n3)Nc4ccc(cc4)C(=O)O,1.0,...,382.142976,0.417797,23.0,6.0,1.0,seed,NaN,NaN,NaN,NaN
78,O14965,AURKA,Aurora kinase A,8C1H,A,A,T3U,4-(4-chloranyl-3-pyrazin-2-yloxy-phenyl)-~{N}-...,Cc1c(cc(c2c1[nH]cc2)c3ccc(c(c3)Oc4cnccn4)Cl)C(...,1.0,...,485.092453,0.427219,22.0,9.0,1.0,seed,NaN,NaN,NaN,NaN


In [16]:
# MaSIF preprocess: split manifest into subsets and submit array job
preprocess_input_subset_dir = os.path.join(masif_preprocess_proc_dir, "input_subsets")
os.makedirs(preprocess_input_subset_dir, exist_ok=True)
preprocess_output_subset_dir = os.path.join(masif_preprocess_proc_dir, "output_subsets")
os.makedirs(preprocess_output_subset_dir, exist_ok=True)

df_preprocess_subsets = np.array_split(df_preprocess_manifest, N_ARRAY_JOBS)
for i, df_subset in enumerate(df_preprocess_subsets):
    df_subset.to_csv(
        os.path.join(preprocess_input_subset_dir, f"input_{i+1}.csv"),
        index=False,
    )

cmd = [
    "sbatch",
    f"--array=1-{N_ARRAY_JOBS}",
    "scripts/slurm/preprocess_array.sh",
    preprocess_input_subset_dir,
    preprocess_output_subset_dir,
]
subprocess.run(cmd, check=True)

/home/ymeng/miniconda3/envs/MaSIF/lib/python3.11/site-packages/numpy/_core/fromnumeric.py:54: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


Submitted batch job 57178552


sbatch: [ESTIMATION] The estimated cost of this job is CHF 2.20
sbatch: ╭──────────────────────────────┬─────────────┬─────────────┬─────────────╮
sbatch: │ [in CHF]                     │ Capping     │ Consumed    │ Queued ¹⁾ ²⁾│
sbatch: ├──────────────────────────────┼─────────────┼─────────────┼─────────────┤
sbatch: │ username : ymeng             │ 0           │ 73.65       │ 2.2         │
sbatch: ├──────────────────────────────┼─────────────┼─────────────┼─────────────┤
sbatch: │ account : upthomae           │ 10,000

CompletedProcess(args=['sbatch', '--array=1-100', 'scripts/slurm/preprocess_array.sh', '/scratch/ymeng/Neosurf_Neosurf/data/processing/2_masif_preprocess/input_subsets', '/scratch/ymeng/Neosurf_Neosurf/data/processing/2_masif_preprocess/output_subsets'], returncode=0)

      │ 88.4        │ 2.2         │
sbatch: ╰──────────────────────────────┴─────────────┴─────────────┴─────────────╯
sbatch: ¹⁾ Estimated cost of the queued jobs and this job
sbatch: ²⁾ Queued jobs costs are based on its walltime (option --time)


In [ ]:
# Gather preprocess output subsets
df_preprocess_results = pd.DataFrame()
for i in range(N_ARRAY_JOBS):
    csv_path = os.path.join(preprocess_output_subset_dir, f"output_{i+1}.csv")
    if os.path.exists(csv_path):
        df_preprocess_results = pd.concat([df_preprocess_results, pd.read_csv(csv_path)])
    else:
        print(f"Warning: {csv_path} does not exist")

print(f"df_preprocess_results.shape: {df_preprocess_results.shape}")

df_preprocess_ok = df_preprocess_results[
    df_preprocess_results["status"].isin(["success", "skipped"])
]
df_preprocess_failed = df_preprocess_results[df_preprocess_results["status"] == "error"]

print(f"Preprocessed successfully or skipped: {df_preprocess_ok.shape[0]}")
print(f"Preprocess errors: {df_preprocess_failed.shape[0]}")
print("Failed entries:")
df_preprocess_failed.head()

___
### Step 2 - Run masif_search around ligand residues
1. Query with VHL structure as target, search for seed patches around the seed ligands

In [27]:
# Get seed_ids from df_seed
def row_to_seed_target(row):
    pdb_id = row.pdb_id
    protein_chain = row.protein_chain
    ligand_chain = row.ligand_chain
    chains = chain_suffix(protein_chain, ligand_chain)
    target = f"{pdb_id}_{chains}"
    return target

seed_ids = df_seed.apply(row_to_seed_target, axis=1).tolist()
seed_ids[0:5]

['6M90_CA', '6M91_CA', '6M92_CA', '6M93_CA', '7AFW_A']

In [ ]:
from pathlib import Path
import subprocess

def submit_neosurf_search(
    seed_ids,
    query_target,
    masif_search_out_dir,
    n_subsets=500,
    dry_run=True
):
    import shutil
    import os

    query_out_dir = os.path.join(masif_search_out_dir, query_target)
    subset_dir = Path(query_out_dir) / "subset"

    # Clear the subset directory before writing new seeds
    if subset_dir.exists():
        shutil.rmtree(subset_dir)
    subset_dir.mkdir(parents=True, exist_ok=True)

    # Evenly split all seed ids across N_SUBSET files (one masif_search.py call per file)
    chunks = [[] for _ in range(n_subsets)]
    for i, seed_id in enumerate(seed_ids):
        chunks[i % n_subsets].append(seed_id)

    for chunk_ix, chunk in enumerate(chunks, start=1):
        if chunk:
            (subset_dir / str(chunk_ix)).write_text("\n".join(chunk) + "\n")

    n_subsets_actual = sum(1 for chunk in chunks if chunk)

    # Clamp n_subsets to number of seeds
    n_subsets_to_use = min(n_subsets_actual, len(seed_ids))

    subset_dir_abs = os.path.abspath(subset_dir)
    print(f"Query target: {query_target}")
    print(f"Wrote {len(seed_ids)} seed(s) into {n_subsets_to_use} subset file(s) under {subset_dir}")
    submit_command = (
        f"sbatch --array=1-{n_subsets_to_use} scripts/slurm/search_array.sh "
        f"{query_target} {os.path.abspath(query_out_dir)} {subset_dir_abs}"
    )
    print(f"Submit: {submit_command}")

    if not dry_run:
        subprocess.run(submit_command, shell=True)

# Example usage:
submit_neosurf_search(seed_ids, query_target="4TZ4_C", masif_search_out_dir=masif_search_out_dir, n_subsets=500, dry_run=True)

Query target: 4TZ4_C
Wrote 26 seed(s) into 26 subset file(s) under /scratch/ymeng/NNeosurf/masif-neosurf/data/masif_search/4TZ4_C/subset
Submit: sbatch --array=1-26 scripts/slurm/search_array.sh 4TZ4_C /scratch/ymeng/NNeosurf/masif-neosurf/data/masif_search/4TZ4_C /scratch/ymeng/NNeosurf/masif-neosurf/data/masif_search/4TZ4_C/subset


In [18]:
# Read df_results from clustered_matches/*.csv (empty if no hits)
from search_output import gather_clustered_results

results_csv = os.path.join(query_out_dir, f"{QUERY_TARGET}_search_results.csv")
df_results = gather_clustered_results(query_out_dir, QUERY_TARGET)

df_results.to_csv(results_csv, index=False)
print(f"Wrote {len(df_results)} matches to {results_csv}")
df_results.head()

FileNotFoundError: [Errno 2] No such file or directory: '/scratch/ymeng/NNeosurf/masif-neosurf/data/masif_search/4TZ4_C/4TZ4_C/clustered_matches'

In [6]:
# Slice max to each unique combination of target, matched_protein, cluster_id with the highest score
df_dedup = df_results.sort_values(by="score", ascending=False).groupby(["target", "matched_protein", "cluster_id"]).first().reset_index()
dedup_csv = os.path.join(query_out_dir, f"{QUERY_TARGET}_dedup.csv")

df_dedup.to_csv(dedup_csv, index=False)
print(f"Wrote {len(df_dedup)} deduplicated matches to {dedup_csv}")
df_dedup.sort_values(by="cluster_size", ascending=False).head()


Wrote 9 deduplicated matches to /scratch/ymeng/NNeosurf/masif-neosurf/data/masif_search/6H0F_B/6H0F_B_dedup.csv


,target,matched_protein,cluster_id,target_site,target_vix,matched_patch_id,score,desc_dist_score,clashing_ca,clashing_heavy,matched_vix,desc_dist,iface_score,mean_desc_dist_score,flattened_transform,cluster_size,cluster_mean_rmsd
1,6H0F_B,5QSV_D,0.0,site_32,4571,29,0.989063,22.453463,0,1,2937,2.741130,0.617374,0.163894,"-0.8848374941118574,-0.459948308188821,-0.0742...",14.0,1.237126
7,6H0F_B,8VLB_A,0.0,site_25,3002,84,0.969552,18.143378,0,1,1350,2.473306,0.913592,0.181434,"0.6532627885401022,-0.7281182894266949,0.20758...",10.0,1.337936
0,6H0F_B,5QSQ_B,0.0,site_9,6300,40,0.977580,24.485135,0,3,2361,2.276409,0.321404,0.172431,"0.18811655710121694,0.3201757470768955,0.92849...",3.0,0.689014
2,6H0F_B,6M92_CA,0.0,site_19,410,58,0.962938,10.208669,0,0,136,3.242997,0.199412,0.086514,"0.34474053707697766,0.7252827786602123,-0.5959...",2.0,0.223082
3,6H0F_B,6M92_CA,1.0,site_15,4631,183,0.935450,9.652207,0,4,429,3.357862,0.249105,0.086957,"-0.054155842613027305,-0.8787217692945033,-0.4...",1.0,0.000000


In [ ]:
df_dedup = pd.read_csv("/scratch/ymeng/NNeosurf/masif-neosurf/data/crbn_ligands_dedup.csv")

# Filter to matched_protein == 7AFW_A
df_dedup = df_dedup[df_dedup["matched_protein"] == "7AFW_A"]

df_dedup

,target,matched_protein,cluster_id,target_site,target_vix,matched_patch_id,score,desc_dist_score,clashing_ca,clashing_heavy,matched_vix,desc_dist,iface_score,mean_desc_dist_score,flattened_transform,cluster_size,cluster_mean_rmsd
9,6H0F_B,7AFW_A,0.0,site_14,6293,80,0.942224,15.276341,0,0,1727,3.394696,0.339545,0.126251,"-0.7859344656941732,-0.5043491230556201,-0.357...",2.0,0.940848
15,6H0G_B,7AFW_A,0.0,site_16,5809,51,0.957798,15.400022,0,4,997,2.579964,0.725148,0.121260,"0.8182982172028582,-0.553642071949644,-0.15449...",1.0,0.000000
28,7LPS_B,7AFW_A,0.0,site_4,6544,70,0.980430,16.071958,0,5,1505,3.076337,0.415029,0.123630,"-0.4173865744442265,-0.006861740615185213,0.90...",2.0,1.120578
43,8D80_B,7AFW_A,0.0,site_24,6528,116,0.954398,16.064220,0,2,2196,2.724984,0.673387,0.127494,"0.1884563509994785,-0.8813487849738003,-0.4332...",5.0,0.704426
59,8RQC_D,7AFW_A,0.0,site_1,1092,146,0.934944,14.659870,0,3,2845,2.966870,0.523973,0.128595,"-0.07851018323045547,-0.5165221746457802,-0.85...",1.0,0.000000
66,8U16_A,7AFW_A,0.0,site_23,5929,140,0.954060,14.906924,0,3,2648,3.278316,0.658839,0.139317,"0.1841958868433828,0.8642898450997718,0.468054...",2.0,0.884970
71,8U17_A,7AFW_A,0.0,site_0,6767,31,0.904493,15.149368,0,2,537,2.909446,0.612826,0.149994,"0.15643916781143524,-0.9365911839426351,-0.313...",1.0,0.000000
75,8UH6_B,7AFW_A,0.0,site_64,2424,42,0.946749,19.012935,0,3,995,2.221536,0.619639,0.157132,"-0.4464701864356142,0.8491355264702862,-0.2821...",1.0,0.000000


In [4]:
def print_pymol_commands(row):
    matched_protein = row["matched_protein"]
    matched_pdb = matched_protein.split("_")[0]
    matched_chains = matched_protein.split("_")[1]
    matched_patch_id = row["matched_patch_id"]

    target_protein = row["target"]
    target_pdb = target_protein.split("_")[0]
    target_chains = target_protein.split("_")[1]
    flattened_transform = row["flattened_transform"]

    # Split matched_chains (e.g. "AB") into ["A", "B"]
    matched_chains = list(matched_chains)
    # Joint matched_chains into a string (e.g. "chain A chain B")
    matched_chains_str = "chain " + " chain ".join(matched_chains)

    # Split target_chains (e.g. "AB") into ["A", "B"]
    target_chains = list(target_chains)
    target_chains_str = "chain " + " chain ".join(target_chains)

    # Load target protein
    print(f"fetch {target_pdb}, {target_protein}_{matched_protein}_{matched_patch_id}")
    print(f"remove {target_protein}_{matched_protein}_{matched_patch_id} AND (not {target_chains_str})")

    # load and transform matched protein
    print(f"fetch {matched_pdb}, {matched_protein}_{matched_patch_id}")
    print(f"remove {matched_protein}_{matched_patch_id} AND (not {matched_chains_str})")
    print(f"apply_transform {matched_protein}_{matched_patch_id}, '{flattened_transform}'")

    # Copy transformed matched_protein to target protein object
    print(f"copy_to {target_protein}_{matched_protein}_{matched_patch_id}, {matched_protein}_{matched_patch_id}")

    # Delete {matched_protein}_{matched_patch_id} object
    print(f"delete {matched_protein}_{matched_patch_id}")

# Apply to all rows
df_dedup.apply(print_pymol_commands, axis=1)

fetch 6H0F, 6H0F_B_7AFW_A_80
remove 6H0F_B_7AFW_A_80 AND (not chain B)
fetch 7AFW, 7AFW_A_80
remove 7AFW_A_80 AND (not chain A)
apply_transform 7AFW_A_80, '-0.7859344656941732,-0.5043491230556201,-0.35768558498636877,-3.6363662083943424,0.11463107809434332,-0.6873130377377955,0.717259021616719,-111.81487360429189,-0.6075909245281254,0.5227167016928679,0.597997088790896,33.085055390718644,0.0,0.0,0.0,1.0'
copy_to 6H0F_B_7AFW_A_80, 7AFW_A_80
delete 7AFW_A_80
fetch 6H0G, 6H0G_B_7AFW_A_51
remove 6H0G_B_7AFW_A_51 AND (not chain B)
fetch 7AFW, 7AFW_A_51
remove 7AFW_A_51 AND (not chain A)
apply_transform 7AFW_A_51, '0.8182982172028582,-0.553642071949644,-0.1544942843277068,-72.9274821077419,0.46224671737335316,0.4741008655351706,0.7493706303134404,35.656988642990825,-0.341637234504941,-0.6846231265930978,0.6438751234002663,-102.12152938615822,0.0,0.0,0.0,1.0'
copy_to 6H0G_B_7AFW_A_51, 7AFW_A_51
delete 7AFW_A_51
fetch 7LPS, 7LPS_B_7AFW_A_70
remove 7LPS_B_7AFW_A_70 AND (not chain B)
fetch 7AFW,

9     None
15    None
28    None
43    None
59    None
66    None
71    None
75    None
dtype: object

In [43]:
def print_pymol_commands(row):
    matched_protein = row["matched_protein"]
    matched_pdb = matched_protein.split("_")[0]
    matched_chains = matched_protein.split("_")[1]
    matched_patch_id = row["matched_patch_id"]
    flattened_transform = row["flattened_transform"]

    # Split matched_chains (e.g. "AB") into ["A", "B"]
    matched_chains = list(matched_chains)
    # Joint matched_chains into a string (e.g. "chain A chain B")
    matched_chains_str = "chain " + " chain ".join(matched_chains)

    print(f"fetch {matched_pdb}, {matched_protein}_{matched_patch_id}")
    #print(f"select {matched_protein}_{matched_patch_id} AND (not {matched_chains_str})")
    #print('cmd.remove("sele");cmd.delete("sele")')
    print(f"apply_transform {matched_protein}_{matched_patch_id}, '{flattened_transform}'")

# Apply to all rows
df_dedup.apply(print_pymol_commands, axis=1)

fetch 5QSQ, 5QSQ_B_40
apply_transform 5QSQ_B_40, '0.18811655710121694,0.3201757470768955,0.9284932158762048,-63.34899149991804,0.1194347315502052,-0.945812608023888,0.30195008760154796,-21.857826309349356,0.9748576849181194,0.05409252708834745,-0.21616311588539106,-29.57452413756281,0.0,0.0,0.0,1.0'
fetch 5QSV, 5QSV_D_29
apply_transform 5QSV_D_29, '-0.8848374941118574,-0.459948308188821,-0.07423047088689808,-154.06565366628317,0.4264906636583265,-0.8637764209055722,0.2683207194754801,4.137717891846782,-0.18753219143957434,0.20576163024675115,0.96046542295497,101.9424492820446,0.0,0.0,0.0,1.0'
fetch 6M92, 6M92_CA_58
apply_transform 6M92_CA_58, '0.34474053707697766,0.7252827786602123,-0.5959184953286809,-67.98240992873171,-0.5430922038973354,-0.3636905840683252,-0.7568223154254735,-60.38063982437132,-0.7656401375070488,0.5845460204628572,0.2685165354298039,-40.661634546822384,0.0,0.0,0.0,1.0'
fetch 6M92, 6M92_CA_183
apply_transform 6M92_CA_183, '-0.054155842613027305,-0.8787217692945033,

0    None
1    None
2    None
3    None
4    None
5    None
6    None
7    None
8    None
dtype: object